# {{ notebook_title }}

Layer-4 Feature Denoising residuals split by the teacher classifier SVD.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from matplotlib import pyplot as plt

sns.set_theme(style="whitegrid")
RUN_DIRS = [Path(path) for path in {{ run_dirs }}]

In [ ]:
manifests = []
for run_dir in RUN_DIRS:
    path = run_dir / "feature_denoising_subspace_errors" / "manifest.json"
    manifests.append(json.loads(path.read_text()))

pd.DataFrame([
    {
        "ID dataset": manifest["dataset"]["name"],
        "activation dim": manifest["subspace"]["activation_dimension"],
        "decisive dim (k)": manifest["subspace"]["decisive_dimension"],
        "insignificant dim": manifest["subspace"]["insignificant_dimension"],
    }
    for manifest in manifests
] )

In [ ]:
def dataset_group(id_name, dataset_name):
    if dataset_name == f"{id_name}_test":
        return "ID"
    if dataset_name.startswith(("cifar10_", "cifar100_")):
        return "Near-OOD"
    return "Far-OOD"

frames = []
for run_dir, manifest in zip(RUN_DIRS, manifests, strict=True):
    id_name = manifest["dataset"]["name"]
    for item in manifest["artifacts"]:
        artifact_path = run_dir / "feature_denoising_subspace_errors" / item["dataset"] / "student_best.pt"
        artifact = torch.load(artifact_path, map_location="cpu", weights_only=False)
        dataset = item["dataset"]
        for component in ("decisive", "insignificant"):
            for metric in ("reconstruction_error", "relative_improvement"):
                values = artifact[f"{component}_{metric}"].numpy()
                frames.append(pd.DataFrame({
                    "id_dataset": id_name.upper().replace("CIFAR", "CIFAR-"),
                    "dataset": dataset,
                    "distribution": dataset_group(id_name, dataset),
                    "component": component.title(),
                    "metric": metric,
                    "value": values,
                }))

scores = pd.concat(frames, ignore_index=True)
scores.groupby(["id_dataset", "distribution", "dataset", "component", "metric"])["value"].agg(["count", "mean", "median", "std"])

In [ ]:
from sklearn.metrics import roc_auc_score

rows = []
for (id_dataset, component, metric), group in scores.groupby(["id_dataset", "component", "metric"]):
    id_values = group[group.distribution == "ID"].value.to_numpy()
    sign = -1.0 if metric == "reconstruction_error" else 1.0
    for dataset, ood in group[group.distribution != "ID"].groupby("dataset"):
        ood_values = ood.value.to_numpy()
        labels = np.r_[np.ones(id_values.size), np.zeros(ood_values.size)]
        values = sign * np.r_[id_values, ood_values]
        rows.append({
            "ID dataset": id_dataset, "OOD dataset": dataset,
            "component": component, "metric": metric,
            "ROC-AUC": roc_auc_score(labels, values),
        })

pd.DataFrame(rows).pivot_table(
    index=["ID dataset", "OOD dataset", "metric"],
    columns="component", values="ROC-AUC",
).round(3)

In [ ]:
error_scores = scores[scores.metric == "reconstruction_error"].copy()
error_scores["log10_error"] = np.log10(error_scores.value.clip(lower=1e-12))
g = sns.displot(
    data=error_scores, x="log10_error", hue="dataset",
    col="component", row="id_dataset", kind="kde",
    common_norm=False, facet_kws={"sharex": False, "sharey": False},
    height=3.2, aspect=1.35,
)
g.set_axis_labels("log10 reconstruction MSE", "density")
g.set_titles("{row_name} ID | {col_name}")

In [ ]:
relative = scores[scores.metric == "relative_improvement"].copy()
relative["plot_value"] = relative.groupby(
    ["id_dataset", "component", "dataset"]
)["value"].transform(lambda values: values.clip(values.quantile(0.005), values.quantile(0.995)))
g = sns.displot(
    data=relative, x="plot_value", hue="dataset",
    col="component", row="id_dataset", kind="kde",
    common_norm=False, facet_kws={"sharex": False, "sharey": False},
    height=3.2, aspect=1.35,
)
g.set_axis_labels("relative improvement (0.5%-99.5% clipped)", "density")
g.set_titles("{row_name} ID | {col_name}")